# 04 — Manifest Build (MCMIPF on-the-fly loading)

## Purpose
This notebook builds supervised manifests for each site (train/val/test) without materializing per-timestamp graph files on disk.

**Key decisions (frozen for this dataset version)**
- Temporal grid: 10 minutes, UTC
- Forecast horizon: 6 hours (H = 36 steps)
- History length: 4 hours (L = 24 steps)
- Spatial patch: 32 × 32 centered at each site
- Graph connectivity: 8-neighborhood on the patch grid

**The manifest stores only:**
- label timestamp (t_label)
- target timestamp (t_target = t_label + H steps)
- target value y (ground GHI at t_target)
- history timestamps (history_ts) for an L-step lookback window ending at t_label

## Import and settings

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()

GROUND_DIR = PROJECT_ROOT / "data" / "ground_aligned"
UNI_GROUND = GROUND_DIR / "ground_10min_utc_uniandes.parquet"
ELP_GROUND = GROUND_DIR / "ground_10min_utc_elpaso.parquet"

MCMIPF_ROOT = PROJECT_ROOT / "data_processed" / "GOES_v2" / "MCMIPF"

META_PATH = PROJECT_ROOT / "data" / "metadata" / "site_center_pix_256.json"

OUT_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("UNI_GROUND:", UNI_GROUND.exists(), "->", UNI_GROUND)
print("ELP_GROUND:", ELP_GROUND.exists(), "->", ELP_GROUND)
print("MCMIPF_ROOT:", MCMIPF_ROOT.exists(), "->", MCMIPF_ROOT)
print("META_PATH:", META_PATH.exists(), "->", META_PATH)
print("OUT_ROOT:", OUT_ROOT)

GROUND UNI: True -> /srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_uniandes.parquet
GROUND ELP: True -> /srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_elpaso.parquet
MCMIPF ROOT: True -> /srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF


## Loading datasets

In [ ]:
uni = pd.read_parquet(UNI_GROUND)
elp = pd.read_parquet(ELP_GROUND)

assert "ghi" in uni.columns and "ghi" in elp.columns, "Missing 'ghi' in ground datasets"
assert str(uni.index.tz) == "UTC", "UNI index is not UTC"
assert str(elp.index.tz) == "UTC", "ELP index is not UTC"

print("UNI:", uni.shape, "|", uni.index.min(), "→", uni.index.max())
print("ELP:", elp.shape, "|", elp.index.min(), "→", elp.index.max())

UNI: (82657, 6) | 2023-09-01 05:00:00+00:00 → 2025-03-28 05:00:00+00:00
ELP: (107172, 9) | 2022-02-21 18:00:00+00:00 → 2024-03-06 23:50:00+00:00


## Temporal splits

In [ ]:
SPLITS = {
    "uniandes": {
        "train": ("2023-09-01", "2024-09-30"),
        "val":   ("2024-10-01", "2024-12-31"),
        "test":  ("2025-01-01", "2025-03-28"),
    },
    "elpaso": {
        "train": ("2022-03-01", "2023-06-30"),
        "val":   ("2023-07-01", "2023-10-31"),
        "test":  ("2023-11-01", "2024-03-07"),
    },
}

def slice_by_date(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC")
    return df.loc[(df.index >= start_ts) & (df.index < end_ts)].copy()

def build_splits(df: pd.DataFrame, site_key: str) -> dict[str, pd.DataFrame]:
    spec = SPLITS[site_key]
    return {name: slice_by_date(df, s, e) for name, (s, e) in spec.items()}

uni_splits = build_splits(uni, "uniandes")
elp_splits = build_splits(elp, "elpaso")

for k, v in uni_splits.items():
    print("UNI", k, v.index.min(), "→", v.index.max(), "| rows:", len(v))
for k, v in elp_splits.items():
    print("ELP", k, v.index.min(), "→", v.index.max(), "| rows:", len(v))

UNI train 2023-09-01 05:00:00+00:00 → 2024-09-29 23:50:00+00:00 | rows: 56850
UNI val 2024-10-01 00:00:00+00:00 → 2024-12-30 23:50:00+00:00 | rows: 13104
UNI test 2025-01-01 00:00:00+00:00 → 2025-03-27 23:50:00+00:00 | rows: 12384
ELP train 2022-03-01 00:00:00+00:00 → 2023-06-29 23:50:00+00:00 | rows: 69984
ELP val 2023-07-01 00:00:00+00:00 → 2023-10-30 23:50:00+00:00 | rows: 17568
ELP test 2023-11-01 00:00:00+00:00 → 2024-03-06 23:50:00+00:00 | rows: 18288


## Temporal parameters

In [ ]:
FREQ_MIN = 10

HOURS_AHEAD = 6
H = int((HOURS_AHEAD * 60) / FREQ_MIN)  # 36

HIST_HOURS = 4
L = int((HIST_HOURS * 60) / FREQ_MIN)   # 24

print("Horizon steps H:", H, f"({HOURS_AHEAD}h)")
print("History steps  L:", L, f"({HIST_HOURS}h)")

Horizon steps H: 36 (6h)
History steps  L: 24 (4h)


## Spatial parameters and site centers

- Patch size is fixed at 32x32 on the 256x256 grid.


In [ ]:
assert META_PATH.exists(), f"Missing metadata JSON: {META_PATH}"

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

GRID_SIZE = int(meta["grid_out"])
SITE_CENTER_PIX = {k: (int(v["row"]), int(v["col"])) for k, v in meta["sites"].items()}

print("GRID_SIZE:", GRID_SIZE)
print("SITE_CENTER_PIX:")
for site, rc in SITE_CENTER_PIX.items():
    print(" ", site, "->", rc)

Loaded SITE_CENTER_PIX:
  uniandes: (row=160, col=49)
  elpaso: (row=91, col=53)


## MCMIPF

MCMIPF files are stored hourly with naming:

    YYYYMMDD_HH_MCMIPF.npz

Each file contains the array "mcmipf" with shape: (6, 16, 256, 256)

corresponding to 6 ten-minute frames within that hour, 16 channels, 256x256 grid.

In [ ]:
def list_mcmipf_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*_MCMIPF.npz"))

FILES = list_mcmipf_files(MCMIPF_ROOT)
print("Found MCMIPF files:", len(FILES))
print("Example:", FILES[0] if FILES else None)

RX = re.compile(r"(?P<ymd>\d{8})_(?P<h>\d{2})_MCMIPF\.npz$")

def parse_hour_key(p: Path) -> tuple[str, int] | None:
    m = RX.search(p.name)
    if not m:
        return None
    return (m.group("ymd"), int(m.group("h")))

hourkey_to_path: dict[tuple[str, int], Path] = {}
bad = 0
for p in FILES:
    key = parse_hour_key(p)
    if key is None:
        bad += 1
        continue
    if key not in hourkey_to_path:
        hourkey_to_path[key] = p

print("Parsed hour-container files:", len(hourkey_to_path))
print("Unparsed files:", bad)

Found MCMIPF files: 27436
Example: /srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF/2022/02/20220201_00_MCMIPF.npz
Parsed hour-container files: 27436
Unparsed files: 0


In [ ]:
def ts_to_hourkey_and_slot(t: pd.Timestamp) -> tuple[tuple[str, int], int]:
    ymd = t.strftime("%Y%m%d")
    hh = int(t.strftime("%H"))
    slot = int(t.strftime("%M")) // 10  # 0..5
    return (ymd, hh), slot


### Coverage check
Timestamp → hourkey + slot (10-min grid)

In [ ]:
def coverage_check(df: pd.DataFrame, label: str) -> None:
    ok = 0
    total = len(df)
    for t in df.index:
        key, slot = ts_to_hourkey_and_slot(t)
        if (key in hourkey_to_path) and (0 <= slot <= 5):
            ok += 1
    print(f"{label} satellite coverage: {ok}/{total} ({100*ok/total:.2f}%)")

coverage_check(uni, "UNI")
coverage_check(elp, "ELP")

UNI satellite coverage: 81553/82657 (98.66%)
ELP satellite coverage: 106050/107172 (98.95%)


## Manifest construction

Each manifest row corresponds to a supervised sample:
 - t_label: current time
 - history_files: list of graph files for the history window [t-L+1, ..., t]
 - y: ground ghi at t_target = t + H steps

In [ ]:
def manifest_for_split(site: str, df_split: pd.DataFrame) -> pd.DataFrame:
    idx = df_split.index
    rows = []

    for t in idx:
        t_target = t + pd.Timedelta(minutes=FREQ_MIN * H)
        if t_target not in idx:
            continue

        y = df_split.loc[t_target, "ghi"]
        if pd.isna(y):
            continue

        t_hist_start = t - pd.Timedelta(minutes=FREQ_MIN * (L - 1))
        hist = pd.date_range(t_hist_start, t, freq=f"{FREQ_MIN}min", tz="UTC")
        if len(hist) != L:
            continue

        # ensure history is inside the split and covered by satellite files
        ok = True
        for th in hist:
            if th not in idx:
                ok = False
                break
            key, slot = ts_to_hourkey_and_slot(th)
            if (key not in hourkey_to_path) or not (0 <= slot <= 5):
                ok = False
                break
        if not ok:
            continue

        rows.append({
            "site": site,
            "t_label": t,
            "t_target": t_target,
            "y": float(y),
            "history_ts": list(hist.astype("datetime64[ns]").astype(str)),
        })

    return pd.DataFrame(rows)

def build_manifests_for_site(site: str, splits: dict[str, pd.DataFrame], out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    # Store metadata required downstream (site center, patch settings, etc.)
    meta_out = {
        "site": site,
        "grid_size": GRID_SIZE,
        "site_center_pix": {"row": int(SITE_CENTER_PIX[site][0]), "col": int(SITE_CENTER_PIX[site][1])},
        "freq_min": FREQ_MIN,
        "horizon_steps": H,
        "history_steps": L,
        "mcmipf_root": str(MCMIPF_ROOT),
        "notes": "Manifest-only dataset. Satellite patches are extracted on-the-fly by the model."
    }
    with open(out_dir / "dataset_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta_out, f, indent=2)

    for split_name, df_split in splits.items():
        man = manifest_for_split(site, df_split)
        out_path = out_dir / f"manifest_{split_name}.parquet"
        man.to_parquet(out_path, index=False)
        print(f"[{site}] {split_name}: {man.shape} -> {out_path}")

## Run: build graph stores and manifests

In [ ]:
uni_out = OUT_ROOT / "uniandes"
elp_out = OUT_ROOT / "elpaso"

build_manifests_for_site("uniandes", uni_splits, uni_out)
build_manifests_for_site("elpaso", elp_splits, elp_out)